# Notebook 00 — Context

Self-healing starter notebook for `thinkthoughts/accurate-intelligence`.

This notebook clones the repo, fixes common GitHub/Colab setup issues, installs the package, loads one arXiv paper, and saves the first context artifacts.

## 0. Clone repo and patch project files

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/thinkthoughts/accurate-intelligence.git"
REPO_DIR = Path("/content/accurate-intelligence")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}

%cd /content/accurate-intelligence

ROOT = Path.cwd()
SRC_DIR = ROOT / "src" / "accurate_intelligence"
SRC_DIR.mkdir(parents=True, exist_ok=True)

(ROOT / "requirements.txt").write_text("""pandas>=2.3.0
numpy>=2.3.0
jupyter>=1.1.0
ipykernel>=6.30.0
matplotlib>=3.10.0
jinja2>=3.1.0
markdown>=3.8.0
pymupdf>=1.26.0
arxiv>=2.2.0
requests>=2.32.0
tqdm>=4.67.0
python-dotenv>=1.1.0
setuptools>=70.0.0
""", encoding="utf-8")

(ROOT / "setup.py").write_text("""from setuptools import setup, find_packages

setup(
    name="accurate-intelligence",
    version="0.0.1",
    packages=find_packages(where="src"),
    package_dir={"": "src"},
)
""", encoding="utf-8")

print("Repo cloned and setup files patched.")

## 1. Patch/verify `src/accurate_intelligence`

In [ ]:
from pathlib import Path
import textwrap

ROOT = Path("/content/accurate-intelligence")
SRC_DIR = ROOT / "src" / "accurate_intelligence"
SRC_DIR.mkdir(parents=True, exist_ok=True)

files = {'__init__.py': '"""Accurate Intelligence: notebook-first research utilities."""\n\nfrom .schemas import Paper, Claim, Evidence, Figure, LabReport, Evaluation\n', 'schemas.py': 'from dataclasses import dataclass, field\nfrom typing import List, Dict\n\n@dataclass\nclass Claim:\n    text: str\n    evidence: List[str] = field(default_factory=list)\n    figure_ids: List[str] = field(default_factory=list)\n\n@dataclass\nclass Evidence:\n    text: str\n    source: str = ""\n\n@dataclass\nclass Figure:\n    identifier: str\n    caption: str = ""\n\n@dataclass\nclass Paper:\n    title: str\n    authors: List[str] = field(default_factory=list)\n    abstract: str = ""\n    sections: Dict[str, str] = field(default_factory=dict)\n\n@dataclass\nclass Evaluation:\n    overall: float = 0.0\n    scientific_core: float = 0.0\n    evidence: float = 0.0\n    protocol: float = 0.0\n    cgcs: float = 0.0\n\n@dataclass\nclass LabReport:\n    context: str = ""\n    observations: List[str] = field(default_factory=list)\n    prescriptions: List[str] = field(default_factory=list)\n', 'paths.py': 'from pathlib import Path\n\nPACKAGE_ROOT = Path(__file__).resolve().parent\nPROJECT_ROOT = PACKAGE_ROOT.parent.parent.parent\n\nFIGURES = PROJECT_ROOT / "figures"\nRESULTS = PROJECT_ROOT / "results"\nREPORTS = PROJECT_ROOT / "reports"\nSITE = PROJECT_ROOT / "site"\n', 'utils.py': 'import json\nfrom pathlib import Path\n\ndef ensure_dir(path):\n    Path(path).mkdir(parents=True, exist_ok=True)\n\ndef save_json(data, path):\n    ensure_dir(Path(path).parent)\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(data, f, indent=2)\n\ndef load_json(path):\n    with open(path, encoding="utf-8") as f:\n        return json.load(f)\n', 'papers.py': 'from .schemas import Paper\n\ndef load_arxiv(arxiv_id: str) -> Paper:\n    try:\n        import arxiv\n    except ImportError as exc:\n        raise ImportError("Missing dependency: arxiv. Run `pip install arxiv`.") from exc\n\n    search = arxiv.Search(id_list=[arxiv_id])\n    client = arxiv.Client()\n    result = next(client.results(search))\n    return Paper(\n        title=result.title,\n        authors=[str(a) for a in result.authors],\n        abstract=result.summary,\n    )\n', 'extraction.py': 'from .schemas import Paper, Claim\n\ndef extract_claims(paper: Paper):\n    claims = []\n    for sentence in paper.abstract.replace("\\n", " ").split("."):\n        sentence = sentence.strip()\n        if sentence:\n            claims.append(Claim(text=sentence))\n    return claims\n\ndef extract_sections(text: str):\n    return {"raw_text": text}\n', 'figures.py': 'from pathlib import Path\nimport matplotlib.pyplot as plt\n\ndef save_figure(fig, path):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, bbox_inches="tight")\n\ndef blank_figure(title="Figure"):\n    fig, ax = plt.subplots()\n    ax.set_title(title)\n    ax.axis("off")\n    return fig\n', 'evaluation.py': 'from .schemas import Evaluation, LabReport\n\ndef score_scientific_accuracy(report: LabReport) -> float:\n    return min(100.0, 20.0 + len(report.observations) * 10.0)\n\ndef score_cgcs(report: LabReport) -> float:\n    has_context = 1.0 if report.context else 0.0\n    has_observations = min(1.0, len(report.observations) / 5)\n    has_prescriptions = min(1.0, len(report.prescriptions) / 3)\n    return round((has_context + has_observations + has_prescriptions) / 3, 3)\n\ndef evaluate_report(report: LabReport) -> Evaluation:\n    accuracy = score_scientific_accuracy(report)\n    cgcs = score_cgcs(report)\n    return Evaluation(\n        overall=accuracy,\n        scientific_core=accuracy,\n        evidence=accuracy,\n        protocol=accuracy,\n        cgcs=cgcs,\n    )\n', 'reporting.py': 'from pathlib import Path\nfrom .schemas import LabReport\n\ndef build_markdown(report: LabReport) -> str:\n    lines = ["# Lab Report", "", "## Context", report.context, ""]\n    lines += ["## Observations"]\n    lines += [f"- {o}" for o in report.observations]\n    lines += ["", "## Prescriptions"]\n    lines += [f"- {p}" for p in report.prescriptions]\n    return "\\n".join(lines)\n\ndef save_markdown_report(report: LabReport, path):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(build_markdown(report), encoding="utf-8")\n'}

for name, content in files.items():
    path = SRC_DIR / name
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")

print("Patched src files:")

!find src/accurate_intelligence -maxdepth 1 -type f | sort

## 2. Install package

In [ ]:
%cd /content/accurate-intelligence

!pip uninstall -y accurate-intelligence -q || true
!pip install -q -r requirements.txt
!pip install -q -e .

print("Installed accurate-intelligence in editable mode.")

## 3. Imports

In [ ]:
import accurate_intelligence
print("Package file:", accurate_intelligence.__file__)

from accurate_intelligence.papers import load_arxiv
from accurate_intelligence.extraction import extract_claims
from accurate_intelligence.schemas import LabReport
from accurate_intelligence.reporting import build_markdown, save_markdown_report
from accurate_intelligence.evaluation import evaluate_report
from accurate_intelligence.utils import ensure_dir, save_json
from accurate_intelligence.paths import RESULTS, REPORTS

ensure_dir(RESULTS)
ensure_dir(REPORTS)

print("Imports successful.")

## 4. Load paper

In [ ]:
ARXIV_ID = "2606.07591"

paper = load_arxiv(ARXIV_ID)

print("Title:", paper.title)
print("Authors:", ", ".join(paper.authors[:8]))
if len(paper.authors) > 8:
    print(f"... plus {len(paper.authors) - 8} more")

print("\nAbstract:\n")
print(paper.abstract)

## 5. Extract initial claims

In [ ]:
claims = extract_claims(paper)

for i, claim in enumerate(claims, 1):
    print(f"{i}. {claim.text}")

## 6. Build context report

In [ ]:
context = (
    f"{paper.title} evaluates whether AI systems can perform end-to-end "
    "scientific research rather than merely summarize or discuss published papers. "
    "For accurate-intelligence, this establishes the first premise of context: "
    "generated research text must be checked against objective, protocol, evidence, "
    "scientific core, and revision quality."
)

report = LabReport(
    context=context,
    observations=[claim.text for claim in claims[:5]],
    prescriptions=[
        "Notebook 01 should decompose objective, protocol, evidence, figures, and scientific core.",
        "Notebook 23 should introduce CGCS for revision and constraint retention.",
        "Notebook 31 should introduce ResearchClaw-style scientific accuracy scoring.",
    ],
)

print(build_markdown(report))

## 7. Evaluate v0 report

In [ ]:
evaluation = evaluate_report(report)

evaluation_dict = {
    "overall": evaluation.overall,
    "scientific_core": evaluation.scientific_core,
    "evidence": evaluation.evidence,
    "protocol": evaluation.protocol,
    "cgcs": evaluation.cgcs,
}

evaluation_dict

## 8. Save artifacts

In [ ]:
result = {
    "arxiv_id": ARXIV_ID,
    "title": paper.title,
    "authors": paper.authors,
    "abstract": paper.abstract,
    "context": report.context,
    "observations": report.observations,
    "prescriptions": report.prescriptions,
    "evaluation": evaluation_dict,
}

save_json(result, RESULTS / "00_context.json")
save_markdown_report(report, REPORTS / "00_context.md")

print("Saved:")
print(RESULTS / "00_context.json")
print(REPORTS / "00_context.md")

## 9. Handoff

Notebook 01 should read:

```python
from accurate_intelligence.utils import load_json
data = load_json("results/00_context.json")
```

and produce objective, protocol, evidence map, figure/table inventory, and scientific core draft.